In [3]:
from pathlib import Path
import sys

print("Python:", sys.executable)
print("Working directory:", Path.cwd())

assert Path("embedder.py").exists(), "Cannot find embedder.py from the current path"
assert Path("models/Xenova/all-MiniLM-L6-v2/model.onnx").exists(), \
    "Cannot find model.onnx，please run uv run python download.py first"
assert Path("models/Xenova/all-MiniLM-L6-v2/tokenizer.json").exists(), \
    "Cannot find tokenizer.json，please run uv run python download.py first"

print("Environment check passed.")

Python: /Users/ruicongrong/Documents/llm-zoomcamp-2026/homework/hw2-vector-search/.venv/bin/python3
Working directory: /Users/ruicongrong/Documents/llm-zoomcamp-2026/homework/hw2-vector-search
Environment check passed.


In [7]:
!pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 46.5 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.3
    Uninstalling pip-25.3:
      Successfully uninstalled pip-25.3


In [8]:
!pip install numpy tqdm

In [1]:
import sys
import numpy as np
import onnxruntime as ort
from tqdm.auto import tqdm

print("Python:", sys.executable)
print("NumPy:", np.__version__)
print("ONNX Runtime:", ort.__version__)

Python: /Users/ruicongrong/Documents/llm-zoomcamp-2026/homework/hw2-vector-search/.venv/bin/python3
NumPy: 2.2.6
ONNX Runtime: 1.23.2


In [2]:
from embedder import Embedder

embed = Embedder()

print("Embedder loaded successfully.")

Embedder loaded successfully.


In [4]:
import numpy as np
from tqdm.auto import tqdm

from embedder import Embedder

embed = Embedder()

print("Embedder loaded successfully.")

Embedder loaded successfully.


In [5]:
def nearest_numeric_option(value, options):
    return min(options, key=lambda option: abs(float(value) - option))

In [6]:
query_q1 = "How does approximate nearest neighbor search work?"

v = embed.encode(query_q1)

print("Vector shape:", v.shape)
print("First value v[0]:", float(v[0]))
print("Vector norm:", np.linalg.norm(v))

q1_options = [-0.31, -0.02, 0.12, 0.44]
q1_answer = nearest_numeric_option(v[0], q1_options)

print("Q1 choose:", q1_answer)

Vector shape: (384,)
First value v[0]: -0.020582006653762086
Vector norm: 0.9999999999999998
Q1 choose: -0.02


In [7]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

print("Number of lesson documents:", len(documents))
print("First document keys:", documents[0].keys())
print("First filename:", documents[0]["filename"])

assert len(documents) == 72

Number of lesson documents: 72
First document keys: dict_keys(['content', 'filename'])
First filename: 01-agentic-rag/lessons/01-intro.md


In [8]:
target_filename_q2 = (
    "02-vector-search/lessons/07-sqlitesearch-vector.md"
)

target_doc_q2 = next(
    doc for doc in documents
    if doc["filename"] == target_filename_q2
)

document_vector_q2 = embed.encode(target_doc_q2["content"])

similarity_q2 = float(v.dot(document_vector_q2))

print("Target file:", target_doc_q2["filename"])
print("Cosine similarity:", similarity_q2)

q2_options = [0.07, 0.37, 0.68, 0.92]
q2_answer = nearest_numeric_option(similarity_q2, q2_options)

print("Q2 choose:", q2_answer)

Target file: 02-vector-search/lessons/07-sqlitesearch-vector.md
Cosine similarity: 0.36107035629941114
Q2 choose: 0.37


In [9]:
from gitsource import chunk_documents

chunks = chunk_documents(
    documents,
    size=2000,
    step=1000,
)

print("Number of documents:", len(documents))
print("Number of chunks:", len(chunks))
print("Chunk keys:", chunks[0].keys())
print("Example filename:", chunks[0]["filename"])
print("Example start:", chunks[0]["start"])
print("Example length:", len(chunks[0]["content"]))

Number of documents: 72
Number of chunks: 295
Chunk keys: dict_keys(['start', 'content', 'filename'])
Example filename: 01-agentic-rag/lessons/01-intro.md
Example start: 0
Example length: 2000


In [10]:
chunk_texts = [chunk["content"] for chunk in chunks]

batch_size = 32
vector_batches = []

for start_idx in tqdm(
    range(0, len(chunk_texts), batch_size),
    desc="Embedding chunks"
):
    batch = chunk_texts[start_idx:start_idx + batch_size]
    batch_vectors = embed.encode_batch(batch)
    vector_batches.append(batch_vectors)

X = np.vstack(vector_batches)

print("Embedding matrix shape:", X.shape)

assert X.shape[0] == len(chunks)
assert X.shape[1] == 384

Embedding chunks:   0%|          | 0/10 [00:00<?, ?it/s]

Embedding matrix shape: (295, 384)


In [11]:
scores_q3 = X.dot(v)

best_idx_q3 = int(np.argmax(scores_q3))
best_chunk_q3 = chunks[best_idx_q3]

q3_answer = best_chunk_q3["filename"]

print("Best score:", float(scores_q3[best_idx_q3]))
print("Best filename:", q3_answer)
print("Best chunk start:", best_chunk_q3["start"])

Best score: 0.6489015983289264
Best filename: 02-vector-search/lessons/07-sqlitesearch-vector.md
Best chunk start: 1000


In [12]:
top_indices_q3 = np.argsort(scores_q3)[::-1][:5]

for rank, idx in enumerate(top_indices_q3, start=1):
    chunk = chunks[int(idx)]
    print(
        f"{rank}. score={scores_q3[idx]:.4f} | "
        f"{chunk['filename']} | start={chunk['start']}"
    )

1. score=0.6489 | 02-vector-search/lessons/07-sqlitesearch-vector.md | start=1000
2. score=0.5510 | 01-agentic-rag/lessons/05-search.md | start=0
3. score=0.4066 | 04-evaluation/lessons/05-search-metrics.md | start=1000
4. score=0.4062 | 02-vector-search/lessons/04-vector-search.md | start=0
5. score=0.4061 | 06-best-practices/lessons/03-reranking.md | start=0


In [13]:
print(q3_answer)

02-vector-search/lessons/07-sqlitesearch-vector.md


In [14]:
from minsearch import VectorSearch

vector_index = VectorSearch(
    keyword_fields=["filename"]
)

vector_index.fit(X, chunks)

print("Vector index created.")

Vector index created.


In [15]:
query_q4 = "What metric do we use to evaluate a search engine?"

query_vector_q4 = embed.encode(query_q4)

vector_results_q4 = vector_index.search(
    query_vector_q4,
    num_results=5,
)

for rank, doc in enumerate(vector_results_q4, start=1):
    print(
        f"{rank}. {doc['filename']} | "
        f"start={doc.get('start')}"
    )

q4_answer = vector_results_q4[0]["filename"]

print("Q4 choose:", q4_answer)

1. 04-evaluation/lessons/05-search-metrics.md | start=0
2. 04-evaluation/lessons/01-intro.md | start=0
3. 01-agentic-rag/lessons/05-search.md | start=0
4. 04-evaluation/lessons/01-intro.md | start=2000
5. 04-evaluation/lessons/15-next-steps.md | start=0
Q4 choose: 04-evaluation/lessons/05-search-metrics.md


In [16]:
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)

text_index.fit(chunks)

print("Text index created.")

Text index created.


In [17]:
query_q5 = "How do I store vectors in PostgreSQL?"

vector_results_q5 = vector_index.search(
    embed.encode(query_q5),
    num_results=5,
)

text_results_q5 = text_index.search(
    query_q5,
    num_results=5,
)

In [18]:
print("VECTOR RESULTS")

for rank, doc in enumerate(vector_results_q5, start=1):
    print(
        f"{rank}. {doc['filename']} | "
        f"start={doc.get('start')}"
    )

VECTOR RESULTS
1. 02-vector-search/lessons/08-pgvector.md | start=0
2. 02-vector-search/lessons/08-pgvector.md | start=3000
3. 03-orchestration/lessons/05-rag.md | start=2000
4. 02-vector-search/lessons/08-pgvector.md | start=1000
5. 02-vector-search/lessons/08-pgvector.md | start=2000


In [19]:
print("TEXT RESULTS")

for rank, doc in enumerate(text_results_q5, start=1):
    print(
        f"{rank}. {doc['filename']} | "
        f"start={doc.get('start')}"
    )

TEXT RESULTS
1. 02-vector-search/lessons/02-embeddings.md | start=4000
2. 03-orchestration/lessons/05-rag.md | start=1000
3. 02-vector-search/lessons/01-intro.md | start=0
4. 03-orchestration/lessons/05-rag.md | start=0
5. 02-vector-search/lessons/01-intro.md | start=1000


In [20]:
vector_files_q5 = {
    doc["filename"]
    for doc in vector_results_q5
}

text_files_q5 = {
    doc["filename"]
    for doc in text_results_q5
}

only_in_vector_q5 = vector_files_q5 - text_files_q5

print("Files only in vector results:")
for filename in sorted(only_in_vector_q5):
    print(filename)

Files only in vector results:
02-vector-search/lessons/08-pgvector.md


In [22]:
q5_options = [
    "02-vector-search/lessons/01-intro.md",
    "02-vector-search/lessons/02-embeddings.md",
    "02-vector-search/lessons/08-pgvector.md",
    "03-orchestration/lessons/05-rag.md",
]

q5_matches = [
    filename
    for filename in q5_options
    if filename in only_in_vector_q5
]

print("Matching answer options:", q5_matches)

assert len(q5_matches) == 1, (
    "Cannot get the only choice，please check if those two searches both use num_results=5"
)

q5_answer = q5_matches[0]

print("Q5 choose:", q5_answer)

Matching answer options: ['02-vector-search/lessons/08-pgvector.md']
Q5 choose: 02-vector-search/lessons/08-pgvector.md


In [23]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])

            scores[key] = (
                scores.get(key, 0)
                + 1 / (k + rank)
            )

            docs[key] = doc

    ranked_keys = sorted(
        scores,
        key=scores.get,
        reverse=True,
    )

    return [
        docs[key]
        for key in ranked_keys[:num_results]
    ]

In [24]:
query_q6 = "How do I give the model access to tools?"

vector_results_q6 = vector_index.search(
    embed.encode(query_q6),
    num_results=5,
)

text_results_q6 = text_index.search(
    query_q6,
    num_results=5,
)

In [25]:
print("VECTOR RESULTS")

for rank, doc in enumerate(vector_results_q6, start=1):
    print(
        f"{rank}. {doc['filename']} | "
        f"start={doc.get('start')}"
    )

print("\nTEXT RESULTS")

for rank, doc in enumerate(text_results_q6, start=1):
    print(
        f"{rank}. {doc['filename']} | "
        f"start={doc.get('start')}"
    )

VECTOR RESULTS
1. 01-agentic-rag/lessons/01-intro.md | start=2000
2. 04-evaluation/lessons/02-ground-truth.md | start=1000
3. 01-agentic-rag/lessons/16-other-frameworks.md | start=0
4. 01-agentic-rag/lessons/15-frameworks.md | start=2000
5. 01-agentic-rag/lessons/13-function-calling.md | start=4000

TEXT RESULTS
1. 01-agentic-rag/lessons/14-agentic-loop.md | start=0
2. 01-agentic-rag/lessons/13-function-calling.md | start=4000
3. 01-agentic-rag/lessons/13-function-calling.md | start=5000
4. 01-agentic-rag/lessons/13-function-calling.md | start=1000
5. 04-evaluation/lessons/02-ground-truth.md | start=3000


In [26]:
hybrid_results_q6 = rrf(
    [vector_results_q6, text_results_q6],
    k=60,
    num_results=5,
)

print("HYBRID RRF RESULTS")

for rank, doc in enumerate(hybrid_results_q6, start=1):
    print(
        f"{rank}. {doc['filename']} | "
        f"start={doc.get('start')}"
    )

q6_answer = hybrid_results_q6[0]["filename"]

print("Q6 choose:", q6_answer)

HYBRID RRF RESULTS
1. 01-agentic-rag/lessons/13-function-calling.md | start=4000
2. 01-agentic-rag/lessons/01-intro.md | start=2000
3. 01-agentic-rag/lessons/14-agentic-loop.md | start=0
4. 04-evaluation/lessons/02-ground-truth.md | start=1000
5. 01-agentic-rag/lessons/16-other-frameworks.md | start=0
Q6 choose: 01-agentic-rag/lessons/13-function-calling.md


In [27]:
answers = {
    "Q1 - Embedding a query": q1_answer,
    "Q2 - Cosine similarity": q2_answer,
    "Q3 - Manual vector search": q3_answer,
    "Q4 - minsearch vector search": q4_answer,
    "Q5 - Vector but not text": q5_answer,
    "Q6 - Hybrid RRF": q6_answer,
}

print("HOMEWORK 2 ANSWERS")
print("=" * 70)

for question, answer in answers.items():
    print(f"{question}: {answer}")

HOMEWORK 2 ANSWERS
Q1 - Embedding a query: -0.02
Q2 - Cosine similarity: 0.37
Q3 - Manual vector search: 02-vector-search/lessons/07-sqlitesearch-vector.md
Q4 - minsearch vector search: 04-evaluation/lessons/05-search-metrics.md
Q5 - Vector but not text: 02-vector-search/lessons/08-pgvector.md
Q6 - Hybrid RRF: 01-agentic-rag/lessons/13-function-calling.md
